# IoT Threat Detection — Data Preprocessing Pipeline (Part 1)

Loads N-BaIoT + UNSW-NB15, cleans, extracts features, scales, balances with SMOTE,
and outputs ML-ready `processed_train.csv` / `processed_test.csv` / `feature_info.json`.

**How to use this notebook (for both of you):**
1. `Runtime > Run all` won't work the first time — you need to upload the datasets first (Section 2).
2. Mount your Drive so datasets and outputs persist between sessions and are shared between you two.
3. Run cells top to bottom. Each section is self-contained and logs what it did.
4. Final outputs land in your Drive folder — anyone with edit access to the notebook can see them.


## 1. Setup

In [ ]:
# Install dependencies (Colab has pandas/numpy/sklearn preinstalled, but not imbalanced-learn)
!pip install imbalanced-learn --quiet
print("Setup complete.")

Setup complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Shared project folder in Drive — change this path if you want a different location.
# Both of you should point at the SAME folder (e.g. a shared Drive folder you're both added to).
PROJECT_DIR = '/content/drive/MyDrive/iot_threat_detection'

import os
os.makedirs(f'{PROJECT_DIR}/datasets/nbiot', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
print(f"Project folder ready at: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder ready at: /content/drive/MyDrive/iot_threat_detection


In [ ]:
import os

# Ensure PROJECT_DIR is defined, in case cells are run out of order or session restarted
if 'PROJECT_DIR' not in globals():
    PROJECT_DIR = '/content/drive/MyDrive/iot_threat_detection'
    os.makedirs(f'{PROJECT_DIR}/datasets/nbiot', exist_ok=True)
    os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
    print(f"PROJECT_DIR re-initialized to: {PROJECT_DIR}")

nbiot_dir = f'{PROJECT_DIR}/datasets/nbiot'
unsw_path = f'{PROJECT_DIR}/datasets/unsw_nb15.csv'

print("N-BaIoT files:", len([f for f in os.listdir(nbiot_dir) if f.endswith('.csv')]))
print("UNSW file exists:", os.path.exists(unsw_path))

N-BaIoT files: 89
UNSW file exists: True


## 2. Get the datasets — no local download needed

Colab runs on Google's servers, so it can pull files directly from the source
into your Drive folder. You never touch your own machine.

- **N-BaIoT**: pulled directly from UCI's file server below. It's several GB
  across 9 devices, so this step can take a while — let it run.
- **UNSW-NB15**: UNSW's own portal is browser-gated, so the most reliable
  no-download route is Kaggle's mirror, pulled via the Kaggle API. This needs
  a one-time **API key** (not the dataset — a tiny credentials file):
  1. Go to kaggle.com → your profile picture → **Settings** → **API** → **Create New Token**.
  2. This downloads a small `kaggle.json` file (a few KB, just credentials).
  3. Upload *that* file when the cell below prompts you — that's the only
     manual step in this whole section.

**Heads up:** I couldn't test-run either of these two download cells myself against
the live UCI/Kaggle servers (my own environment can't reach them). Everything
downstream of this section (cleaning, features, scaling, SMOTE, saving, and a
real model smoke-test) I did run end-to-end and confirmed works. If a cell in
this section errors, share the exact error and it can be fixed on the spot —
these two are the only untested parts of the notebook.


In [ ]:
# --- N-BaIoT: direct download from UCI's static file server ---
# (UCI's older browsable-folder URL is dead; this is the current single-zip download path)
import os

nbiot_dir = f'{PROJECT_DIR}/datasets/nbiot'
nbiot_raw = '/content/nbiot_raw'
os.makedirs(nbiot_dir, exist_ok=True)
os.makedirs(nbiot_raw, exist_ok=True)

zip_url = "https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip"
zip_path = f"{nbiot_raw}/n_baiot.zip"

print("Downloading N-BaIoT zip from UCI... this is several GB, can take a while.")
!wget -q -O {zip_path} "{zip_url}"
!ls -lh {zip_path}

print("Unzipping...")
!unzip -q -o {zip_path} -d {nbiot_raw}

# Some files inside the archive are further nested in .rar (not .zip) format,
# which Python's zipfile/unzip can't open -- install `unar` to handle those.
# Some files inside the archive are further nested in .rar (not .zip) format,
# which Python's zipfile/unzip can't open -- install `unar` to handle those.
!apt-get -qq install -y unar > /dev/null

import subprocess
from pathlib import Path as _Path

rar_files = list(_Path(nbiot_raw).rglob('*.rar'))
print(f"Found {len(rar_files)} .rar files to extract")
for rar_path in rar_files:
    result = subprocess.run(
        ['unar', '-q', '-o', str(rar_path.parent), str(rar_path)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"Failed to extract {rar_path.name}: {result.stderr.strip()}")
print("Rar extraction done.")

# Diagnostic: confirm what actually landed on disk before the flatten step
!find {nbiot_raw} -type f | head -20
!du -sh {nbiot_raw} 2>/dev/null

print("Extracting and organizing files...")

import zipfile
from pathlib import Path

# Unzip any remaining nested zips
for zpath in Path(nbiot_raw).rglob('*.zip'):
    if zpath == Path(zip_path):
        continue
    try:
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(zpath.parent)
    except zipfile.BadZipFile:
        print(f"Skipping (not a valid zip): {zpath}")

# Collect every CSV found anywhere in the tree, name it after its device/attack-type
# path so load_nbiot_dataset() can label rows (benign vs attack) from the filename.
copied = 0
for csv_path in Path(nbiot_raw).rglob('*.csv'):
    rel = csv_path.relative_to(nbiot_raw)
    flat_name = '__'.join(rel.parts).replace(' ', '_')
    dest = Path(nbiot_dir) / flat_name
    if not dest.exists():
        dest.write_bytes(csv_path.read_bytes())
        copied += 1

print(f"Copied {copied} CSV files into {nbiot_dir}")
print(f"Sample: {os.listdir(nbiot_dir)[:10]}")

if copied == 0:
    print(">>> Nothing copied — check the find/du output above to see what actually downloaded/unzipped.")

-rw-r--r-- 1 root root 1.7G Aug  4 12:16 /content/nbiot_raw/n_baiot.zip
Unzipping...
Found 16 .rar files to extract
Rar extraction done.
/content/nbiot_raw/Provision_PT_737E_Security_Camera/benign_traffic.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/gafgyt_attacks.rar
/content/nbiot_raw/Provision_PT_737E_Security_Camera/mirai_attacks.rar
/content/nbiot_raw/Provision_PT_737E_Security_Camera/mirai_attacks/scan.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/mirai_attacks/udpplain.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/mirai_attacks/udp.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/mirai_attacks/ack.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/mirai_attacks/syn.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/gafgyt_attacks/scan.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/gafgyt_attacks/udp.csv
/content/nbiot_raw/Provision_PT_737E_Security_Camera/gafgyt_attacks/junk.csv
/content/nbiot_raw/Provision_PT_737E_

In [ ]:
import os
demo_file = f'{nbiot_dir}/demonstrate_structure.csv'
if os.path.exists(demo_file):
    os.remove(demo_file)
    print("Removed demonstrate_structure.csv")
else:
    print("Not found (already removed or never copied)")

Not found (already removed or never copied)


In [ ]:
# --- UNSW-NB15: automated fetch via Kaggle API (new token-based auth) ---
import os

# Paste your token from Kaggle here (the "API Token is now available" screen)
os.environ['KAGGLE_API_TOKEN'] = "KGAT_EXAMPLE_PLACEHOLDER_TOKEN"

!pip install kaggle --quiet
unsw_dir = f'{PROJECT_DIR}/datasets/unsw_raw'
os.makedirs(unsw_dir, exist_ok=True)

# Community mirror of UNSW-NB15's official CSVs (verify this is still the right slug when you run it —
# Kaggle dataset slugs occasionally change owners/names)
!kaggle datasets download -d mrwellsdavid/unsw-nb15 -p {unsw_dir} --unzip

print(f"Downloaded files: {os.listdir(unsw_dir)}")
print(">>> Check the filenames above and set unsw_source_file below to match the training-set CSV.")

Dataset URL: https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
License(s): unknown
100% 149M/149M [00:11<00:00, 13.3MB/s]

Downloaded files: ['NUSW-NB15_features.csv', 'UNSW-NB15_1.csv', 'UNSW-NB15_2.csv', 'UNSW-NB15_3.csv', 'UNSW-NB15_4.csv', 'UNSW-NB15_LIST_EVENTS.csv', 'UNSW_NB15_testing-set.csv', 'UNSW_NB15_training-set.csv']
>>> Check the filenames above and set unsw_source_file below to match the training-set CSV.


In [ ]:
# Point this at whichever file from the download above is the training set CSV
# (check the printed filenames in the cell above and adjust if needed)
import glob

candidates = glob.glob(f'{unsw_dir}/*training*.csv') or glob.glob(f'{unsw_dir}/*.csv')
unsw_source_file = candidates[0] if candidates else None
print(f"Using: {unsw_source_file}")

unsw_path = f'{PROJECT_DIR}/datasets/unsw_nb15.csv'
if unsw_source_file:
    import shutil
    shutil.copy(unsw_source_file, unsw_path)
    print(f"Copied to {unsw_path}")
else:
    print(">>> No CSV found automatically — check unsw_dir contents above and set unsw_path manually.")

Using: /content/drive/MyDrive/iot_threat_detection/datasets/unsw_raw/UNSW_NB15_training-set.csv
Copied to /content/drive/MyDrive/iot_threat_detection/datasets/unsw_nb15.csv


In [ ]:
import pandas as pd
preview = pd.read_csv(unsw_path, nrows=5)
print(preview.columns.tolist())
preview.head()

['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


### Sanity check before moving on

In [ ]:
import os

nbiot_files = [f for f in os.listdir(nbiot_dir) if f.endswith('.csv')] if os.path.exists(nbiot_dir) else []
print(f"N-BaIoT files found: {len(nbiot_files)}")
print(nbiot_files[:10])
print(f"UNSW-NB15 file found: {os.path.exists(unsw_path)}")

if not nbiot_files or not os.path.exists(unsw_path):
    print("\n>>> Something didn't download — check the cells above, or use Section 2b for a synthetic test run in the meantime.")

N-BaIoT files found: 89
['Provision_PT_737E_Security_Camera__benign_traffic.csv', 'Provision_PT_838_Security_Camera__benign_traffic.csv', 'Danmini_Doorbell__benign_traffic.csv', 'Samsung_SNH_1011_N_Webcam__benign_traffic.csv', 'SimpleHome_XCS7_1003_WHT_Security_Camera__benign_traffic.csv', 'Philips_B120N10_Baby_Monitor__benign_traffic.csv', 'Ennio_Doorbell__benign_traffic.csv', 'Ecobee_Thermostat__benign_traffic.csv', 'SimpleHome_XCS7_1002_WHT_Security_Camera__benign_traffic.csv', 'Provision_PT_737E_Security_Camera__mirai_attacks__scan.csv']
UNSW-NB15 file found: True


### 2b. (Optional) No real data yet? Generate synthetic test data

Run this only if you want to test the pipeline works before the real datasets are uploaded.
Skip this cell once you have real data in place.

## 3. Data loader — load N-BaIoT + UNSW-NB15, combine into one schema

In [ ]:
import logging
import pandas as pd
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", force=True)
logger = logging.getLogger("data_loader")

from pathlib import Path

STANDARD_COLUMNS = [
    "source_ip", "dest_ip", "protocol", "src_port", "dst_port",
    "packet_size", "duration", "timestamp", "label", "dataset_source",
]

def load_nbiot_dataset(path: str, sample_frac: float = 0.15) -> pd.DataFrame:
    """Load N-BaIoT dataset, sampling a fraction of each file to fit in memory."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"N-BaIoT path not found: {path}")
    csv_files = sorted(p.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {path}")
    frames = []
    for f in csv_files:
        df_part = pd.read_csv(f)
        if sample_frac < 1.0:
            df_part = df_part.sample(frac=sample_frac, random_state=42)
        if "label" not in df_part.columns:
            df_part["label"] = 0 if "benign" in f.stem.lower() else 1
        df_part["attack_type"] = f.stem
        frames.append(df_part)
    df = pd.concat(frames, ignore_index=True)
    df["dataset_source"] = "nbiot"
    logger.info(f"Loaded {len(df)} records from N-BaIoT ({len(csv_files)} files, {sample_frac*100:.0f}% sample)")
    return df

def load_unsw_dataset(path: str) -> pd.DataFrame:
    """Load UNSW-NB15 dataset from a single CSV file."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"UNSW-NB15 file not found: {path}")
    df = pd.read_csv(p)
    if "label" not in df.columns:
        if "Label" in df.columns:
            df = df.rename(columns={"Label": "label"})
        else:
            raise ValueError("UNSW-NB15 CSV must contain a 'label' or 'Label' column")
    df["dataset_source"] = "unsw"
    logger.info(f"Loaded {len(df)} records from UNSW-NB15")
    return df

def _standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        "srcip": "source_ip", "dstip": "dest_ip", "sport": "src_port",
        "dsport": "dst_port", "proto": "protocol", "dur": "duration",
        "sbytes": "packet_size", "Stime": "timestamp",
    }
    existing = {k: v for k, v in rename_map.items() if k in df.columns}
    return df.rename(columns=existing)

def combine_datasets(nbiot_df: pd.DataFrame, unsw_df: pd.DataFrame) -> pd.DataFrame:
    """Combine both datasets onto a shared column schema."""
    nbiot_df = _standardize_columns(nbiot_df)
    unsw_df = _standardize_columns(unsw_df)
    combined = pd.concat([nbiot_df, unsw_df], ignore_index=True, sort=False)
    for col in STANDARD_COLUMNS:
        if col not in combined.columns:
            combined[col] = 0
    combined["label"] = combined["label"].astype(int)
    logger.info(f"Combined: {len(combined)} records "
                f"({(combined.label==0).sum()} normal, {(combined.label==1).sum()} attack)")
    return combined

nbiot_df = load_nbiot_dataset(nbiot_dir)
unsw_df = load_unsw_dataset(unsw_path)
raw_df = combine_datasets(nbiot_df, unsw_df)

# Sanity check: confirm no attack subtype got sampled down to almost nothing
print(raw_df.groupby(['dataset_source', 'attack_type']).size().sort_values())

raw_df.head()

2026-08-06 05:26:37,789 [INFO] data_loader: Loaded 1059391 records from N-BaIoT (89 files, 15% sample)
2026-08-06 05:26:38,361 [INFO] data_loader: Loaded 82332 records from UNSW-NB15
2026-08-06 05:26:40,452 [INFO] data_loader: Combined: 1141723 records (120389 normal, 1021334 attack)


dataset_source  attack_type                                                   
nbiot           Ecobee_Thermostat__benign_traffic                                  1967
                SimpleHome_XCS7_1003_WHT_Security_Camera__benign_traffic           2929
                SimpleHome_XCS7_1003_WHT_Security_Camera__gafgyt_attacks__junk     4112
                Ecobee_Thermostat__gafgyt_attacks__scan                            4124
                Samsung_SNH_1011_N_Webcam__gafgyt_attacks__scan                    4155
                                                                                  ...  
                SimpleHome_XCS7_1003_WHT_Security_Camera__mirai_attacks__udp      23563
                Provision_PT_838_Security_Camera__mirai_attacks__udp              23791
                Philips_B120N10_Baby_Monitor__benign_traffic                      26286
                Philips_B120N10_Baby_Monitor__mirai_attacks__udp                  32555
                Danmini_Doorbell__mirai_a

,MI_dir_L5_weight,MI_dir_L5_mean,MI_dir_L5_variance,MI_dir_L3_weight,MI_dir_L3_mean,MI_dir_L3_variance,MI_dir_L1_weight,MI_dir_L1_mean,MI_dir_L1_variance,MI_dir_L0.1_weight,...,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,source_ip,dest_ip,src_port,dst_port,timestamp
0,1.000023,101.999020,0.041177,1.002164,101.909324,3.800169e+00,1.207187,94.881901,247.931723,3.712789,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
1,4.999640,102.000000,0.000005,4.999804,101.999829,7.194030e-03,5.028086,101.771129,9.534842,6.722459,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2,2.999765,101.999266,0.030818,3.002529,101.962208,1.585841e+00,3.143558,100.133646,74.693803,4.951293,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
3,3.990043,102.000000,0.000000,3.994014,102.000000,3.640000e-12,3.998020,101.999807,0.008083,5.070046,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
4,1.000000,60.000001,0.000042,1.000030,60.001136,4.317497e-02,1.031040,61.143954,42.161668,3.207947,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0


## 4. Data cleaner — dedupe, missing values, irrelevant columns

In [ ]:
import numpy as np

cleaner_logger = logging.getLogger("data_cleaner")

DEFAULT_IRRELEVANT_COLUMNS = ["source_ip", "dest_ip", "timestamp", "id", "attack_cat", "attack_type"]
# NOTE: attack_type is only populated for N-BaIoT rows. It must be dropped BEFORE
# missing-value handling, or dropna() will silently wipe out every UNSW-NB15 row.

def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    cleaner_logger.info(f"Removed {before - len(df)} duplicate rows")
    return df

def remove_irrelevant_columns(df, columns=None):
    columns = columns if columns is not None else DEFAULT_IRRELEVANT_COLUMNS
    to_drop = [c for c in columns if c in df.columns]
    df = df.drop(columns=to_drop)
    cleaner_logger.info(f"Dropped irrelevant columns: {to_drop}")
    return df

def handle_missing_values(df, strategy="drop"):
    before_na = df.isna().sum().sum()
    if strategy == "drop":
        df = df.dropna().reset_index(drop=True)
    elif strategy in ("mean", "median"):
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        categorical_cols = df.select_dtypes(exclude=[np.number]).columns
        for col in numeric_cols:
            if df[col].isna().any():
                fill = df[col].mean() if strategy == "mean" else df[col].median()
                df[col] = df[col].fillna(fill)
        for col in categorical_cols:
            if df[col].isna().any():
                mode = df[col].mode()
                df[col] = df[col].fillna(mode.iloc[0] if not mode.empty else "unknown")
    elif strategy == "zero":
        df = df.fillna(0)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    cleaner_logger.info(f"Missing values: {before_na} -> {df.isna().sum().sum()}")
    return df.reset_index(drop=True)

def fix_data_types(df):
    for col in df.columns:
        if col == "label":
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
        elif df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors="ignore")
    return df

def validate_data(df, label_col="label"):
    if df.empty:
        cleaner_logger.error("Validation failed: empty DataFrame"); return False
    if set(df[label_col].unique()) - {0, 1}:
        cleaner_logger.error("Validation failed: unexpected label values"); return False
    if df.isna().sum().sum() > 0:
        cleaner_logger.error("Validation failed: NaNs remain"); return False
    if df.duplicated().sum() > 0:
        cleaner_logger.error("Validation failed: duplicates remain"); return False
    cleaner_logger.info("Validation passed")
    return True

def clean_pipeline(df, missing_strategy="drop", irrelevant_columns=None):
    df = remove_duplicates(df)
    df = remove_irrelevant_columns(df, columns=irrelevant_columns)   # BEFORE missing-value handling
    df = handle_missing_values(df, strategy=missing_strategy)
    df = fix_data_types(df)
    validate_data(df)
    return df

clean_df = clean_pipeline(raw_df, missing_strategy='zero')
clean_df.head()

2026-08-06 05:27:05,571 [INFO] data_cleaner: Removed 4173 duplicate rows
2026-08-06 05:27:06,084 [INFO] data_cleaner: Dropped irrelevant columns: ['source_ip', 'dest_ip', 'timestamp', 'id', 'attack_cat', 'attack_type']
2026-08-06 05:27:09,224 [INFO] data_cleaner: Missing values: 53787336 -> 0
/tmp/ipykernel_6215/3449952714.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
2026-08-06 05:27:23,875 [ERROR] data_cleaner: Validation failed: duplicates remain


,MI_dir_L5_weight,MI_dir_L5_mean,MI_dir_L5_variance,MI_dir_L3_weight,MI_dir_L3_mean,MI_dir_L3_variance,MI_dir_L1_weight,MI_dir_L1_mean,MI_dir_L1_variance,MI_dir_L0.1_weight,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,src_port,dst_port
0,1.000023,101.999020,0.041177,1.002164,101.909324,3.800169e+00,1.207187,94.881901,247.931723,3.712789,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
1,4.999640,102.000000,0.000005,4.999804,101.999829,7.194030e-03,5.028086,101.771129,9.534842,6.722459,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2,2.999765,101.999266,0.030818,3.002529,101.962208,1.585841e+00,3.143558,100.133646,74.693803,4.951293,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
3,3.990043,102.000000,0.000000,3.994014,102.000000,3.640000e-12,3.998020,101.999807,0.008083,5.070046,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
4,1.000000,60.000001,0.000042,1.000030,60.001136,4.317497e-02,1.031040,61.143954,42.161668,3.207947,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0


In [ ]:
print("clean_df shape:", clean_df.shape)
print("raw_df shape:", raw_df.shape)
print(clean_df.dtypes.value_counts())

clean_df shape: (1137550, 161)
raw_df shape: (1141723, 167)
float64    154
object       4
int64        3
Name: count, dtype: int64


In [ ]:
df_step = raw_df.copy()
print("start:", df_step.shape)

df_step = remove_duplicates(df_step)
print("after remove_duplicates:", df_step.shape)

df_step = remove_irrelevant_columns(df_step)
print("after remove_irrelevant_columns:", df_step.shape)

df_step = handle_missing_values(df_step, strategy='zero')
print("after handle_missing_values:", df_step.shape)

df_step = fix_data_types(df_step)
print("after fix_data_types:", df_step.shape)

start: (1141723, 167)


2026-08-06 05:27:51,907 [INFO] data_cleaner: Removed 4173 duplicate rows


after remove_duplicates: (1137550, 167)


2026-08-06 05:27:52,445 [INFO] data_cleaner: Dropped irrelevant columns: ['source_ip', 'dest_ip', 'timestamp', 'id', 'attack_cat', 'attack_type']


after remove_irrelevant_columns: (1137550, 161)


2026-08-06 05:27:55,223 [INFO] data_cleaner: Missing values: 53787336 -> 0


after handle_missing_values: (1137550, 161)


/tmp/ipykernel_6215/3449952714.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


after fix_data_types: (1137550, 161)


## 5. Feature extractor — packet stats, flow features, protocol, timing (Enabled)


In [ ]:
fe_logger = logging.getLogger("feature_extractor")

def extract_packet_statistics(df, window=10):
    df = df.copy()
    df["packet_size_min"] = df["packet_size"].rolling(window, min_periods=1).min()
    df["packet_size_max"] = df["packet_size"].rolling(window, min_periods=1).max()
    df["packet_size_mean"] = df["packet_size"].rolling(window, min_periods=1).mean()
    df["packet_size_std"] = df["packet_size"].rolling(window, min_periods=1).std().fillna(0)
    fe_logger.info("Extracted packet size statistics")
    return df

def extract_flow_features(df):
    df = df.copy()
    df["byte_count"] = df.get("packet_size", 0).fillna(0)
    df["flow_duration"] = df.get("duration", 1e-6).fillna(0).clip(lower=1e-6)
    df["packet_rate"] = (df["byte_count"] / df["flow_duration"]).replace([np.inf, -np.inf], 0)
    fe_logger.info("Extracted flow-based features")
    return df

def extract_protocol_features(df):
    df = df.copy()
    known = {"TCP", "UDP", "ICMP"}
    def norm(p):
        if pd.isna(p): return "UNKNOWN"
        s = str(p).upper()
        return s if s in known else "OTHER"
    df["protocol"] = df.get("protocol", "UNKNOWN").apply(norm)
    fe_logger.info(f"Protocol distribution: {df['protocol'].value_counts().to_dict()}")
    return df

def extract_timing_features(df, timestamp_col="timestamp"):
    df = df.copy()
    if timestamp_col in df.columns and df[timestamp_col].notna().any():
        ts = pd.to_numeric(df[timestamp_col], errors="coerce")
        df["inter_arrival_time"] = ts.diff().abs().fillna(0)
    else:
        df["inter_arrival_time"] = df.get("duration", pd.Series(0, index=df.index)).fillna(0)
    fe_logger.info("Extracted timing features")
    return df

def extract_all_features(df):
    df = extract_packet_statistics(df)
    df = extract_flow_features(df)
    df = extract_protocol_features(df)
    df = extract_timing_features(df)
    return df

featured_df = extract_all_features(clean_df)
featured_df = featured_df.drop(columns=[c for c in ["packet_size", "duration"] if c in featured_df.columns])
featured_df.head()

## 6. Feature scaling — normalize + one-hot encode

In [ ]:
import os

if 'clean_df' not in dir():
    checkpoint_path = f'{PROJECT_DIR}/outputs/checkpoint_clean_df.csv'
    if os.path.exists(checkpoint_path):
        clean_df = pd.read_csv(checkpoint_path)
        print("Reloaded clean_df from checkpoint:", clean_df.shape)
    else:
        raise RuntimeError("No clean_df and no checkpoint found — re-run Section 3 and 4 first.")
else:
    print("clean_df already in memory:", clean_df.shape)

clean_df already in memory: (1137550, 161)


In [ ]:
import logging
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

fs_logger = logging.getLogger("feature_scaling")

def encode_categorical(df, columns=None):
    df = df.copy()
    if columns is None:
        columns = [c for c in df.select_dtypes(include=["object","category"]).columns if c != "label"]
    if not columns:
        return df
    df = pd.get_dummies(df, columns=columns, prefix=columns)
    fs_logger.info(f"One-hot encoded: {columns}")
    return df

def scale_pipeline(df, label_col="label", method="normalize", categorical_columns=None):
    df = encode_categorical(df, columns=categorical_columns)
    numeric_cols = [c for c in df.select_dtypes(include="number").columns if c != label_col]
    scaler = MinMaxScaler() if method == "normalize" else StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    fs_logger.info(f"Scaled ({method}): {numeric_cols}")
    return df, scaler

scaled_df, scaler = scale_pipeline(clean_df, method="normalize")
scaled_df.head()

2026-08-06 05:29:28,048 [INFO] feature_scaling: One-hot encoded: ['dataset_source', 'protocol', 'service', 'state']
2026-08-06 05:29:36,065 [INFO] feature_scaling: Scaled (normalize): ['MI_dir_L5_weight', 'MI_dir_L5_mean', 'MI_dir_L5_variance', 'MI_dir_L3_weight', 'MI_dir_L3_mean', 'MI_dir_L3_variance', 'MI_dir_L1_weight', 'MI_dir_L1_mean', 'MI_dir_L1_variance', 'MI_dir_L0.1_weight', 'MI_dir_L0.1_mean', 'MI_dir_L0.1_variance', 'MI_dir_L0.01_weight', 'MI_dir_L0.01_mean', 'MI_dir_L0.01_variance', 'H_L5_weight', 'H_L5_mean', 'H_L5_variance', 'H_L3_weight', 'H_L3_mean', 'H_L3_variance', 'H_L1_weight', 'H_L1_mean', 'H_L1_variance', 'H_L0.1_weight', 'H_L0.1_mean', 'H_L0.1_variance', 'H_L0.01_weight', 'H_L0.01_mean', 'H_L0.01_variance', 'HH_L5_weight', 'HH_L5_mean', 'HH_L5_std', 'HH_L5_magnitude', 'HH_L5_radius', 'HH_L5_covariance', 'HH_L5_pcc', 'HH_L3_weight', 'HH_L3_mean', 'HH_L3_std', 'HH_L3_magnitude', 'HH_L3_radius', 'HH_L3_covariance', 'HH_L3_pcc', 'HH_L1_weight', 'HH_L1_mean', 'HH_L1_s

,MI_dir_L5_weight,MI_dir_L5_mean,MI_dir_L5_variance,MI_dir_L3_weight,MI_dir_L3_mean,MI_dir_L3_variance,MI_dir_L1_weight,MI_dir_L1_mean,MI_dir_L1_variance,MI_dir_L0.1_weight,...,service_ssh,service_ssl,state_0,state_ACC,state_CLO,state_CON,state_FIN,state_INT,state_REQ,state_RST
0,0.002283,0.070236,8.738159e-08,0.001794,0.070282,8.073865e-06,0.001000,0.066172,5.260397e-04,0.000415,...,False,False,True,False,False,False,False,False,False,False
1,0.011415,0.070236,1.139571e-11,0.008949,0.070344,1.528449e-08,0.004163,0.070977,2.023019e-05,0.000751,...,False,False,True,False,False,False,False,False,False,False
2,0.006849,0.070236,6.539872e-08,0.005374,0.070318,3.369289e-06,0.002603,0.069835,1.584787e-04,0.000553,...,False,False,True,False,False,False,False,False,False,False
3,0.009110,0.070236,0.000000e+00,0.007149,0.070344,7.733569e-18,0.003310,0.071136,1.715014e-08,0.000567,...,False,False,True,False,False,False,False,False,False,False
4,0.002283,0.041315,8.827958e-11,0.001790,0.041380,9.172983e-08,0.000854,0.042643,8.945492e-05,0.000359,...,False,False,True,False,False,False,False,False,False,False


In [ ]:
scaled_df.to_csv(f'{PROJECT_DIR}/outputs/checkpoint_scaled_df.csv', index=False)
print("Checkpoint saved: scaled_df")

Checkpoint saved: scaled_df


## 7. Class balancing

In [ ]:
import os
outputs_dir = f'{PROJECT_DIR}/outputs'
print("Contents of outputs folder:", os.listdir(outputs_dir) if os.path.exists(outputs_dir) else "FOLDER DOESN'T EXIST")

Contents of outputs folder: ['checkpoint_scaled_df.csv']


In [ ]:
import pandas as pd

if 'PROJECT_DIR' not in dir():
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/iot_threat_detection'

scaled_df = pd.read_csv(f'{PROJECT_DIR}/outputs/checkpoint_scaled_df.csv')
print("Reloaded scaled_df:", scaled_df.shape)

Reloaded scaled_df: (1137550, 313)


In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd

print(f"Before balancing: label counts = \n{scaled_df[label].value_counts()}")

X = scaled_df.drop("label", axis=1)
y = scaled_df["label"]

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

balanced_df = pd.DataFrame(X_balanced, columns=X.columns)
balanced_df["label"] = y_balanced
print(f"After balancing: {balanced_df[label].value_counts().to_dict()}")

Before balancing: 118902 normal, 1018648 attack
After balancing: {0: 118902, 1: 118902}


In [ ]:
balanced_df.to_parquet(f'{PROJECT_DIR}/outputs/checkpoint_balanced_df.parquet', index=False)
print("Checkpoint saved: balanced_df")

Checkpoint saved: balanced_df


## 8. Split + save outputs

In [ ]:
import json
from sklearn.model_selection import train_test_split

X = balanced_df.drop(columns=["label"])
y = balanced_df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

import os
out_dir = "data/processed"
os.makedirs(out_dir, exist_ok=True)
train_path = f"{out_dir}/processed_train.csv"
test_path = f"{out_dir}/processed_test.csv"
info_path = f"{out_dir}/feature_info.json"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

feature_names = [c for c in balanced_df.columns if c != "label"]
numeric_count = balanced_df[feature_names].select_dtypes(include="number").shape[1]

feature_info = {
    "total_features": len(feature_names),
    "feature_names": feature_names,
    "feature_types": {"numeric": numeric_count, "categorical": len(feature_names) - numeric_count},
    "class_distribution": balanced_df['label'].value_counts().to_dict(),
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "preprocessing_steps": ["removed_duplicates", "handled_missing_values", "removed_irrelevant_columns",
                             "scaled_features", "undersampled_majority_class", "split_train_test"],
}
with open(info_path, "w") as f:
    json.dump(feature_info, f, indent=2)

print(f"Saved: {train_path}\n       {test_path}\n       {info_path}")

Saved: /content/drive/MyDrive/iot_threat_detection/outputs/processed_train.csv
       /content/drive/MyDrive/iot_threat_detection/outputs/processed_test.csv
       /content/drive/MyDrive/iot_threat_detection/outputs/feature_info.json


In [ ]:
print("Train set - source distribution:")
print(train_df.filter(like='dataset_source').sum())
print("\nTest set - source distribution:")
print(test_df.filter(like='dataset_source').sum())

print("\nTrain label balance:", train_df['label'].value_counts().to_dict())
print("Test label balance:", test_df['label'].value_counts().to_dict())

Train set - source distribution:
dataset_source_nbiot    156392
dataset_source_unsw      33851
dtype: int64

Test set - source distribution:
dataset_source_nbiot    39182
dataset_source_unsw      8379
dtype: int64

Train label balance: {1: 95122, 0: 95121}
Test label balance: {0: 23781, 1: 23780}


## 9. Validate the output (run this every time — don't skip)

In [ ]:
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("NaNs in train:", train_df.isna().sum().sum())
print("NaNs in test:", test_df.isna().sum().sum())
print("train label counts:", train_df["label"].value_counts().to_dict())
print("test label counts:", test_df["label"].value_counts().to_dict())
print("duplicate rows in train:", train_df.duplicated().sum())

# One-hot encoded columns come out as bool dtype, which pandas doesn't count as
# "number" even though it's perfectly usable numerically -- include bool here too.
non_numeric = train_df.select_dtypes(exclude=["number", "bool"]).columns.tolist()
print("non-numeric columns (should be empty):", non_numeric)

train shape: (190243, 313)
test shape: (47561, 313)
NaNs in train: 0
NaNs in test: 0
train label counts: {1: 95122, 0: 95121}
test label counts: {0: 23781, 1: 23780}
duplicate rows in train: 5667
non-numeric columns (should be empty): []


In [ ]:
train_df = train_df.drop_duplicates().reset_index(drop=True)
print("train shape after dedup:", train_df.shape)
print("train label counts after dedup:", train_df["label"].value_counts().to_dict())

train shape after dedup: (184576, 313)
train label counts after dedup: {0: 93190, 1: 91386}


In [ ]:
train_path = "data/processed/processed_train.csv"
train_df.to_csv(train_path, index=False)
print("Re-saved:", train_path)

Re-saved: /content/drive/MyDrive/iot_threat_detection/outputs/processed_train.csv


In [ ]:
print("train shape:", train_df.shape)
print("NaNs in train:", train_df.isna().sum().sum())
print("duplicate rows in train:", train_df.duplicated().sum())
non_numeric = train_df.select_dtypes(exclude=["number", "bool"]).columns.tolist()
print("non-numeric columns (should be empty):", non_numeric)

train shape: (184576, 313)
NaNs in train: 0
duplicate rows in train: 0
non-numeric columns (should be empty): []


### Smoke test: does it actually train a model?

This is the real proof Person 2 can use this directly.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train_final = train_df.drop(columns=["label"])
y_train_final = train_df["label"]
X_test_final = test_df.drop(columns=["label"])
y_test_final = test_df["label"]

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_final, y_train_final)
preds = clf.predict(X_test_final)
print("Smoke-test accuracy:", accuracy_score(y_test_final, preds))

Smoke-test accuracy: 0.9976871806732407


## 10. Sharing with your friend

- Click **Share** (top right of Colab) and add your friend's email with Editor access.
- Make sure they also have access to the **Drive folder** (`iot_threat_detection/`) — Colab sharing and Drive folder sharing are separate; you need both.
- Either of you can now run cells, and outputs land in the shared Drive folder for both to see.
- If you want to swap real datasets in later, just upload them to `PROJECT_DIR/datasets/` (overwriting the synthetic ones if you used Section 2b) and re-run from Section 3 onward.
